In [1]:

import numpy as np
import sys
from pathlib import Path
import os
currentdir = os.getcwd()
ind = currentdir.find("HLS4ML_VS_MANUAL")
root_folder = currentdir[0:ind+len("HLS4ML_VS_MANUAL")]
path = os.path.join(root_folder, "src/hdl/Batchnorm-JetTagging")
sys.path.insert(1,path)
import helper_functions as f
sys.path.insert(1,"/home/caleb/HLS4ML_VS_MANUAL/src/hdl/Batchnorm-JetTagging/")
from helper_functions import dec_to_bin
def char_to_int(char) -> int:
    try:
        return int(char)
    except:
        lookup = "ABCDEF"
        return 10+lookup.find(char.upper())
def hex_to_dec(number) -> int:
    output = 0
    current_n = len(number) - 1
    for char in number:
        output+= (char_to_int(char))*(16**current_n)
        current_n-=1
    return output
def hex_to_dec_signed(number) -> int:
    output = 0
    neg = char[0]=="1"
    current_n = len(number) - 2
    for char in number[1:]:
        output+= (char_to_int(char))*(16**current_n)
        current_n-=1
    return -1*output if neg else 1
def dec_to_hex(number: int, bits: int) -> str:
    string = ""
    while (number!=0):
        res = number%16
        if (res>=10):
            res="ABCDEF"[res-10]
        string=f"{res}{string}"
        number=int(number/16)
    return string


def dec_to_base(number: int, base: int) -> str:
    string = ""
    while (number!=0):
        res = number%base
        if (res>=10):
            res="ABCDEFGHIJKLMNOPQRSTUVWXYZ"[res-10]
        string=f"{res}{string}"
        number=int(number/base)
    return string


In [2]:
import numpy as np
def table_gen(eq, name, width_lookup, nfrac_lookup, width, nfrac, signedMem = True, signedLookup = True):
    if signedLookup:
        # Minimum value for lookup
        MIN=-2**(width_lookup-1-nfrac_lookup)
        # Maximum value for lookup
        MAX=2**(width_lookup-1-nfrac_lookup) 
    else:
        # Since it isn't signed, minimum is 0
        MIN=0
        MAX=2**(width_lookup-nfrac_lookup)-2**(-nfrac_lookup)
    print(MAX)
    # Number of values in table
    length = 2**width_lookup
    # Lookup values to put into eq
    ind=np.linspace(MIN, MAX, length)
    # The values to be put in the table
    arr = eq(ind)
    if (signedLookup):
        split = 2**(width_lookup-1)
        arr=np.concatenate((arr[split:],arr[0:split]))
    sig_int = 1 if signedMem else 0
    max_val=(2**(width-sig_int)-1)/(2**nfrac)
    with open(f"{name}_table_{width}_{nfrac}_{width_lookup}_{nfrac_lookup}.dat", "w") as f:
        for elem in arr:
            elem = min(elem, max_val)
            # If there's no sign, tell the function it's wider then chop off the sign
            item = dec_to_bin(int(elem*(2**nfrac)), width+1-sig_int)
            f.write(item[1-sig_int:])
            f.write("\n")

In [ ]:
def tanh(x):
    expon = np.exp(2*x)
    return (expon-1)/(expon+1)
table_gen(tanh, "tanh", width_lookup=10, nfrac_lookup=6, width=18, nfrac=18, signedMem=0, signedLookup=0)

7.9921875


In [ ]:
def sigmoid(x):
    return 1/(1+np.exp(-x))-.5
table_gen(sigmoid, "sigmoid", width_lookup=10, nfrac_lookup=6, width=18, nfrac=18, signedMem=False, signedLookup=False)

7.9921875
